# Train ACT on ALOHA Transfer Cube (Google Colab)

Part of [vla-robustness-aloha](https://github.com/bobbyzare/vla-robustness-aloha), Phase 2.

Trains an ACT policy on the human demonstrations `lerobot/aloha_sim_transfer_cube_human`
with the same settings as the official `lerobot/act_aloha_sim_transfer_cube_human`, so the
two can be compared. Evaluation for the README is done separately (50 episodes, seeds 1000-1049).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Have your Hugging Face token (type **Write**) and your W&B API key ready.
   Paste them only into the login boxes below. They are never saved in this notebook.

**Order**
1. Run cells 1-5 once per session (cell 2 takes a few minutes).
2. Run cell 6 with `MODE = "test"` first (about 1,000 steps). It checks install, logins,
   training, checkpoint upload, W&B and evaluation, and shows the training speed.
3. Then set `MODE = "full"` in cell 5, run cell 5 again, and run cell 6.
4. If Colab disconnects during the full run, start a new session, run cells 1-5 and then cell 7.

## 1. Check the GPU

Training needs a CUDA GPU (a T4 has 15 GB of memory).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install LeRobot in a Python 3.12 environment

Colab's own Python may be newer than 3.12. The ALOHA simulator needs `labmaze`
(through gym-aloha and dm-control), which has ready-made packages only up to Python 3.12.
So LeRobot is installed into a separate Python 3.12 environment, the same version as the
local machine. `uv` downloads Python 3.12 and installs the packages; no root access needed.
The notebook itself (logins, checks) keeps running on Colab's Python.

LeRobot is pinned to the same commit used for evaluation on the local machine, so training
and evaluation run the same code (checkpoint formats change between LeRobot versions).
Extras: `training` (W&B, accelerate) and `aloha` (gym-aloha, for evaluation during training).

In [ ]:
LEROBOT_COMMIT = "8c920c4270460851cedd2737657584586d3dc66f"
VENV = "/content/venv312"
PY = f"{VENV}/bin/python"                    # Python 3.12 with LeRobot
LEROBOT_TRAIN = f"{VENV}/bin/lerobot-train"

!pip install -q uv
!uv venv {VENV} --python 3.12
!uv pip install --python {PY} "lerobot[training,aloha] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}"

## 3. Set up headless rendering and check the install

`MUJOCO_GL=egl` draws the simulator's camera images on the GPU without a screen,
the same setting as on the local machine. Commands started from this notebook inherit it.

The check runs in the Python 3.12 environment: it imports LeRobot, creates the ALOHA
environment and renders one camera image (a black image would mean rendering is broken).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

CHECK = '''
import sys, torch, lerobot, gymnasium as gym, gym_aloha
env = gym.make("gym_aloha/AlohaTransferCube-v0", obs_type="pixels_agent_pos")
obs, _ = env.reset(seed=0)
img = obs["pixels"]["top"]
print("python", sys.version.split()[0], "| lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("rendered image", img.shape, "mean brightness", round(float(img.mean()), 1), "(about 40 is correct, about 0 means black)")
'''
with open("/content/check_setup.py", "w") as f:
    f.write(CHECK)
!{PY} /content/check_setup.py

## 4. Log in to Hugging Face and Weights & Biases

Each call shows a login box. Paste the token or key there. The logins are stored in the
Colab session (not in this notebook), and the Python 3.12 environment uses them too.
- Hugging Face: token of type **Write**, so checkpoints can be uploaded.
- W&B: API key from wandb.ai (user settings → API keys).

In [ ]:
!pip install -q wandb huggingface_hub
from huggingface_hub import login, whoami
import wandb

login()                      # Hugging Face
print("Hugging Face user:", whoami()["name"])
wandb.login()                # Weights & Biases

## 5. Settings

`MODE = "test"`: short run into a separate private repo, to check the whole pipeline.
`MODE = "full"`: the real run, same settings as the official pretrained model.

Settings shared by both modes, and why:
- `steps`, `batch_size`, `seed`: as in the official model's `train_config.json`.
- `save_freq`: a checkpoint every 10,000 steps (default 20,000), so a disconnect loses less.
- `save_checkpoint_to_hub`: upload every checkpoint to `checkpoints/<step>/` in the model repo
  while training runs; Colab files are lost when the session ends.
- `push_to_hub`: at the end, upload the final model to the repo root, loadable as `--policy.path=<repo>`.
- Evaluation during training: 10 episodes every 20,000 steps, logged to W&B with a video.
  It is only a progress check; the reported result comes from the separate 50-episode evaluation.
- `wandb.disable_artifact`: do not also upload each checkpoint to W&B (it is already on the Hub).
- `eval.use_async_envs=false`: run the evaluation environments in the main process. With the
  default (`true`), each environment runs in a fresh worker process that has not imported
  gym_aloha, and creating the environment fails with "Namespace gym_aloha not found".

In [ ]:
MODE = "test"                 # "test" first, then "full"

HF_USER = whoami()["name"]    # your Hugging Face username, from the login in cell 4
SETTINGS = {
    "test": dict(repo=f"{HF_USER}/act_aloha_transfer_cube_test", private=True,
                 steps=1_000, save_freq=500, env_eval_freq=1_000, eval_episodes=2),
    "full": dict(repo=f"{HF_USER}/act_aloha_transfer_cube", private=False,
                 steps=100_000, save_freq=10_000, env_eval_freq=20_000, eval_episodes=10),
}[MODE]

JOB_NAME = SETTINGS["repo"].split("/")[1]
OUTPUT_DIR = f"/content/outputs/train/{JOB_NAME}"
print(MODE, SETTINGS, OUTPUT_DIR, sep="\n")

## 6. Train

Progress is printed every 200 steps (`loss`, `updt_s` = seconds per step). Live charts are on
wandb.ai, project `vla-robustness-aloha`.

Keep this browser tab open during the full run. If the output folder already exists,
lerobot-train stops with an error. For the full run, use cell 7 to resume instead. To repeat a
crashed test run, delete its folder first: `!rm -rf /content/outputs/train/act_aloha_transfer_cube_test`

In [ ]:
s = SETTINGS
train_args = f"""
--policy.type=act
--dataset.repo_id=lerobot/aloha_sim_transfer_cube_human
--env.type=aloha --env.task=AlohaTransferCube-v0
--steps={s['steps']} --batch_size=8 --seed=1000 --num_workers=2
--save_freq={s['save_freq']} --env_eval_freq={s['env_eval_freq']}
--eval.n_episodes={s['eval_episodes']} --eval.batch_size={s['eval_episodes']} --eval.use_async_envs=false
--policy.device=cuda
--policy.repo_id={s['repo']} --policy.private={str(s['private']).lower()} --policy.push_to_hub=true
--save_checkpoint_to_hub=true
--wandb.enable=true --wandb.project=vla-robustness-aloha --wandb.disable_artifact=true
--output_dir={OUTPUT_DIR} --job_name={JOB_NAME}
""".split()
TRAIN_ARGS = " ".join(train_args)
print("lerobot-train", TRAIN_ARGS)
!{LEROBOT_TRAIN} {TRAIN_ARGS}

## 7. Resume after a disconnect (only if needed)

Downloads the newest `checkpoints/<step>/` from the model repo and continues training from
that step, with the settings stored in the checkpoint. Run cells 1-5 first, with the same `MODE`.

In [ ]:
REPO = SETTINGS["repo"]
!{LEROBOT_TRAIN} --config_path={REPO} --resume=true

## 8. Check what is on the Hub

In [ ]:
from huggingface_hub import HfApi
files = HfApi().list_repo_files(SETTINGS["repo"])
print(len(files), "files in", SETTINGS["repo"])
print("checkpoints:", sorted({f.split("/")[1] for f in files if f.startswith("checkpoints/")}))
print("final model at repo root:", "model.safetensors" in files)